# Mani-Mamba Geometry Interpretability Plots (ETTm2)

Expects data from `run_explain_ETTm2.sh`.

In [ ]:
folder = "ETTm2_96_96_ManiMamba_GEOMETRY_EXPLAIN_l3_itr0"

import json
import os

import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = os.path.abspath(folder)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Liberation Sans", "Arial"],
    "font.size": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "figure.dpi": 300,
})

VAR_NAMES = ["HUFL", "HULL", "MUFL", "MULL", "LUFL", "LULL", "OT"]
D = len(VAR_NAMES)
TRIU_LABELS = []
for i in range(D):
    for j in range(i, D):
        TRIU_LABELS.append(f"({VAR_NAMES[i]},{VAR_NAMES[j]})")
TRI_DIM = D * (D + 1) // 2
NPRIME_LABELS = VAR_NAMES + ["month", "day", "weekday", "hour", "minute"]

with open(os.path.join(DATA_DIR, "meta.json")) as f:
    META = json.load(f)

N_LAYERS = META["e_layers"]
D_STATE = META["d_state"]
COV_WINDOW = META["cov_window"]
COV_STRIDE = META["cov_stride"]
SEQ_LEN = META["seq_len"]
T_W = (SEQ_LEN - COV_WINDOW) // COV_STRIDE + 1
LAST_W = T_W - 1
LAST_W_START = LAST_W * COV_STRIDE
LAST_W_END = LAST_W_START + COV_WINDOW - 1

print(f"DATA_DIR: {DATA_DIR}")
print(f"Meta: {json.dumps(META, indent=2)}")
print(f"Derived: T_W={T_W}, LAST_W={LAST_W}, TRI_DIM={TRI_DIM}, N_LAYERS={N_LAYERS}, D_STATE={D_STATE}")

## Figure 1: Covariance Matrices + Tangent Vector

In [ ]:
cov = np.load(os.path.join(DATA_DIR, "spd_cov.npy"))
v_t = np.load(os.path.join(DATA_DIR, "spd_v_t.npy"))

sigma_first = cov[0, 0]
sigma_last = cov[0, LAST_W]
v_last = v_t[0, LAST_W].reshape(1, -1)

CMAP = "RdBu_r"
COL_W = 7.16

fig = plt.figure(figsize=(COL_W, COL_W * 0.58))
gs = fig.add_gridspec(
    2, 2,
    height_ratios=[1, 0.45],
    hspace=0.38,
    wspace=0.35,
    left=0.10, right=0.94, top=0.93, bottom=0.10,
)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[1, :])

for ax, mat, title, vr in [
    (ax_a, sigma_first, rf"(a) Covariance matrix at early window ($t \in [0,{COV_WINDOW - 1}]$)", 1.5),
    (ax_b, sigma_last, rf"(b) Covariance matrix at late window ($t \in [{LAST_W_START},{LAST_W_END}]$)", None),
]:
    v = vr if vr is not None else abs(mat).max()
    im = ax.imshow(mat, aspect="equal", interpolation="nearest",
                    cmap=CMAP, vmin=-1.0 if vr is None else -v, vmax=v)
    ax.set_xticks(range(D))
    ax.set_xticklabels(VAR_NAMES, fontsize=6, rotation=45, ha="right")
    ax.set_yticks(range(D))
    ax.set_yticklabels(VAR_NAMES, fontsize=6)
    ax.set_title(title, fontsize=9)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(labelsize=5.5)

v_abs = abs(v_last).max()
im_c = ax_c.imshow(v_last, aspect="auto", interpolation="nearest",
                    cmap=CMAP, vmin=-v_abs, vmax=v_abs)
ax_c.set_yticks([])
ax_c.set_xticks(range(TRI_DIM))
ax_c.set_xticklabels(TRIU_LABELS, fontsize=4.5, rotation=45, ha="right")
ax_c.set_title(rf"(c) Tangent vector $v_{{{LAST_W}}}$: rate of covariance change at window {LAST_W}", fontsize=9)
ax_c.set_xlabel("Covariance component (upper-triangular)", fontsize=7)
cb_c = fig.colorbar(im_c, ax=ax_c, fraction=0.012, pad=0.015)
cb_c.ax.tick_params(labelsize=5.5)

plt.show()

## Figure 2: SPD Geodesic Distance + Geometry-modulated SSM Projections

In [ ]:
from mpl_toolkits.axes_grid1 import make_axes_locatable

log_cov = np.load(os.path.join(DATA_DIR, "spd_log_cov.npy"))
lc = log_cov[0]
flat = lc.reshape(T_W, -1)

dist = np.zeros((T_W, T_W))
for i in range(T_W):
    for j in range(T_W):
        dist[i, j] = np.linalg.norm(flat[i] - flat[j])

last_li = N_LAYERS - 1
geo_files = [f"layer{last_li}_fwd_geo_b", f"layer{last_li}_fwd_geo_c"]
arrays = []
for fname in geo_files:
    path = os.path.join(DATA_DIR, f"{fname}.npy")
    if os.path.exists(path):
        arrays.append(np.load(path)[0])
    else:
        raise FileNotFoundError(f"{path} not found")

arr_b, arr_c = arrays
sel_labels = ["HUFL", "LUFL", "day", "minute"]
sel_idx = [NPRIME_LABELS.index(lb) for lb in sel_labels]
arr_b_sel = arr_b[:, sel_idx]
arr_c_sel = arr_c[:, sel_idx]

y_tick_step = max(1, D_STATE // 4)

fig = plt.figure(figsize=(7.16, 2.8))

# Left: geodesic distance | Right: nested 2-row for shared title + 2 BC subplots
gs_outer = fig.add_gridspec(
    1, 2,
    width_ratios=[1.2, 1],
    wspace=0.25,
    left=0.07, right=0.95, top=0.92, bottom=0.22,
)
ax_dist = fig.add_subplot(gs_outer[0, 0])

gs_right = gs_outer[0, 1].subgridspec(
    2, 2,
    height_ratios=[0.06, 1],
    hspace=0.15,
    wspace=0.65,
)
ax_title = fig.add_subplot(gs_right[0, :])
ax_title.set_axis_off()
ax_title.text(0.5, 2.2, "Geometry-Modulated Projection", fontsize=9,
            ha="center", va="top", transform=ax_title.transAxes)

ax_b = fig.add_subplot(gs_right[1, 0])
ax_c = fig.add_subplot(gs_right[1, 1])

n_ticks = 5
ticks = np.linspace(0, T_W - 1, n_ticks, dtype=int)
im_dist = ax_dist.imshow(dist, aspect="equal", interpolation="nearest", cmap="magma")
ax_dist.set_xticks(ticks)
ax_dist.set_yticks(ticks)
ax_dist.set_xticklabels([f"$w_{{{i}}}$" for i in ticks], fontsize=7)
ax_dist.set_yticklabels([f"$w_{{{i}}}$" for i in ticks], fontsize=7)
ax_dist.set_xlabel("Window index", fontsize=8)
ax_dist.set_ylabel("Window index", fontsize=8)
ax_dist.set_title("SPD Geodesic Distance Matrix", fontsize=9)
cb_dist = fig.colorbar(im_dist, ax=ax_dist, fraction=0.046, pad=0.04)
cb_dist.set_label(r"$\|\log \Sigma_i - \log \Sigma_j\|_F$", fontsize=7.5)
cb_dist.ax.tick_params(labelsize=6)

for ax, arr, lbl in [(ax_b, arr_b_sel, r"$B_{\mathrm{geo}}$"), (ax_c, arr_c_sel, r"$C_{\mathrm{geo}}$")]:
    v_abs = abs(arr).max()
    im = ax.imshow(arr, aspect="auto", interpolation="nearest",
                    cmap="RdBu_r", vmin=-v_abs, vmax=v_abs)
    ax.set_xticks(range(len(sel_labels)))
    ax.set_xticklabels(sel_labels, fontsize=6, rotation=45, ha="right")
    ax.set_yticks(range(0, D_STATE, y_tick_step))
    ax.set_yticklabels([str(s) for s in range(0, D_STATE, y_tick_step)], fontsize=6.5)
    ax.set_ylabel("State dim", fontsize=7)
    ax.set_title(lbl, fontsize=9)
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.05)
    cb = fig.colorbar(im, cax=cax)
    cb.ax.tick_params(labelsize=6)

plt.show()


## Figure 3: Combined Covariance + Geodesic Distance (Paper Figure)

In [ ]:
fig = plt.figure(figsize=(COL_W, COL_W * 0.3))
gs = fig.add_gridspec(
    1, 3,
    width_ratios=[1, 1, 1.3],
    wspace=0.4,
    left=0.07, right=0.96, top=0.88, bottom=0.18,
)
ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_d = fig.add_subplot(gs[0, 2])

for ax, mat, title, vr in [
    (ax_a, sigma_first, rf"(a) Covariance ($t \in [0,{COV_WINDOW - 1}]$)", 1.5),
    (ax_b, sigma_last, rf"(b) Covariance ($t \in [{LAST_W_START},{LAST_W_END}]$)", None),
]:
    v = vr if vr is not None else abs(mat).max()
    im = ax.imshow(mat, aspect="equal", interpolation="nearest",
                    cmap=CMAP, vmin=-1.0 if vr is None else -v, vmax=v)
    ax.set_xticks(range(D))
    ax.set_xticklabels(VAR_NAMES, fontsize=5, rotation=45, ha="right")
    ax.set_yticks(range(D))
    ax.set_yticklabels(VAR_NAMES, fontsize=5)
    ax.set_title(title, fontsize=8)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(labelsize=5)

n_ticks = 5
ticks = np.linspace(0, T_W - 1, n_ticks, dtype=int)
im_d = ax_d.imshow(dist, aspect="equal", interpolation="nearest", cmap="magma")
ax_d.set_xticks(ticks)
ax_d.set_yticks(ticks)
ax_d.set_xticklabels([f"$w_{{{i}}}$" for i in ticks], fontsize=6)
ax_d.set_yticklabels([f"$w_{{{i}}}$" for i in ticks], fontsize=6)
ax_d.set_xlabel("Window index", fontsize=7)
ax_d.set_ylabel("Window index", fontsize=7)
ax_d.set_title("(c) SPD Geodesic Distance", fontsize=8)
cb_d = fig.colorbar(im_d, ax=ax_d, fraction=0.046, pad=0.04)
cb_d.set_label(r"$\|\log \Sigma_i - \log \Sigma_j\|_F$", fontsize=6.5)
cb_d.ax.tick_params(labelsize=5)

plt.show()